# Colab Experiment: CLIP-space Concept Poisoning on Caltech-101

Goal: use natural 224x224 object images for CLIP-space concept poisoning and semantic-shift metrics.

## 1. Setup Colab / GitHub repo

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ngocvuq4/adversarial-data-protection.git"
PROJECT_DIR = "adversarial-data-protection"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("src").exists():
    if not Path(PROJECT_DIR).exists():
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    os.chdir(PROJECT_DIR)

print("Working directory:", os.getcwd())
print("Installing: requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Optional Google Drive dataset cache

In [ ]:
# Google Drive dataset/results paths.
# Your Drive folder is: MyDrive/adversarial-data-protection/
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/adversarial-data-protection"
DRIVE_DATA_ROOT = f"{DRIVE_PROJECT_DIR}/data"
DRIVE_RESULTS_DIR = f"{DRIVE_PROJECT_DIR}/results"

DATA_ROOT = "./data"
RESULTS_ROOT = "./results"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_ROOT = DRIVE_DATA_ROOT
        RESULTS_ROOT = DRIVE_RESULTS_DIR
        Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
        Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)
    except ImportError:
        print("Not running in Colab; using local ./data and ./results")

print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)


## 3. Run experiment

In [ ]:
import os
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from scripts.run_experiment import setup_dirs, collect_clean_tensors
from src.datasets import get_caltech101
from src.evaluation import compute_attack_success_rate, compute_linf, compute_psnr, compute_ssim
from src.models import evaluate, get_victim_resnet18, train_one_epoch
from src.visualization import plot_before_after

from src.techniques.nightshade import _normalize_clip, get_text_embedding, load_clip_model, poison_images

# Caltech-101 natural images are better suited for CLIP-space semantic shift than CIFAR 32x32.
SUBSET_SIZE = 500
BATCH_SIZE = 16
IMG_SIZE = 224
EPSILON = 0.05
TARGET_CONCEPT = "a photo of a cat"
BASELINE_EPOCHS = 2
VICTIM_EPOCHS = 2
PGD_STEPS = 8
LEARNING_RATE = 0.01
WEIGHT_DECAY = 5e-4
RUN_NAME = f"caltech101_subset{SUBSET_SIZE}_eps{EPSILON}_target_{TARGET_CONCEPT.replace(' ', '_')}_base{BASELINE_EPOCHS}_victim{VICTIM_EPOCHS}_pgd{PGD_STEPS}"
RUN_DIR = Path("results") / "concept_poisoning" / RUN_NAME
TABLE_DIR = RUN_DIR / "tables"
SAMPLE_DIR = RUN_DIR / "samples"
TENSOR_DIR = RUN_DIR / "tensors"
MODEL_DIR = RUN_DIR / "models"
FIGURE_DIR = RUN_DIR / "figures"
for path in [TABLE_DIR, SAMPLE_DIR, TENSOR_DIR, MODEL_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)


def train_classifier_with_history(model_fn, loader, epochs, device, lr=LEARNING_RATE):
    model = model_fn().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    history = []
    for epoch in range(epochs):
        loss, acc = train_one_epoch(model, loader, optimizer, criterion, device)
        history.append({"epoch": epoch + 1, "train_loss": loss, "train_accuracy": acc, "lr": lr})
        print(f"  epoch {epoch + 1}/{epochs} loss={loss} train_acc={acc}")
    return model, history


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
setup_dirs()

train_loader, test_loader, num_classes = get_caltech101(
    root=DATA_ROOT,
    img_size=IMG_SIZE,
    subset_size=SUBSET_SIZE,
    batch_size=BATCH_SIZE,
    download=True,
)
clean_x, clean_y = collect_clean_tensors(train_loader)
print("num_classes:", num_classes, "train_images:", len(clean_y))

print("Training clean baseline victim on clean Caltech-101...")
baseline, baseline_history = train_classifier_with_history(
    lambda: get_victim_resnet18(num_classes=num_classes, device=device),
    train_loader,
    BASELINE_EPOCHS,
    device,
)
baseline_acc = evaluate(baseline, test_loader, device)
print("baseline_clean_test_accuracy:", baseline_acc)

print("Generating Caltech-101 concept-poisoned train set at 224x224...")
clip_model, _ = load_clip_model("ViT-B/32", device)
protected_batches = []
for start in range(0, clean_x.size(0), BATCH_SIZE):
    batch = clean_x[start:start+BATCH_SIZE].to(device)
    protected = poison_images(
        clip_model,
        batch,
        target_concept=TARGET_CONCEPT,
        epsilon=EPSILON,
        pgd_steps=PGD_STEPS,
        device=device,
    )
    protected_batches.append(protected.detach().cpu())
    del batch, protected
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
protected_x = torch.cat(protected_batches, dim=0).clamp(0, 1)

torch.save(
    {
        "x_clean": clean_x.cpu(),
        "x_protected": protected_x.cpu(),
        "y": clean_y.cpu(),
        "epsilon": EPSILON,
        "target_concept": TARGET_CONCEPT,
        "technique": "concept_poisoning",
        "dataset": "Caltech-101",
        "subset_size": SUBSET_SIZE,
        "run_name": RUN_NAME,
    },
    TENSOR_DIR / "protected_dataset.pt",
)

@torch.no_grad()
def compute_clip_target_shift(x_orig, x_protected, target_text, batch_size=16):
    target = get_text_embedding(clip_model, target_text, device)
    before_values, after_values = [], []
    clip_model.eval()
    for start in range(0, x_orig.size(0), batch_size):
        xo = x_orig[start:start+batch_size].to(device)
        xp = x_protected[start:start+batch_size].to(device)
        fo = F.normalize(clip_model.encode_image(_normalize_clip(xo)).float(), dim=1)
        fp = F.normalize(clip_model.encode_image(_normalize_clip(xp)).float(), dim=1)
        before_values.append(F.cosine_similarity(fo, target.expand_as(fo)).cpu())
        after_values.append(F.cosine_similarity(fp, target.expand_as(fp)).cpu())
    before = torch.cat(before_values)
    after = torch.cat(after_values)
    return round(before.mean().item(), 4), round(after.mean().item(), 4), round((after - before).mean().item(), 4)

clip_before, clip_after, clip_delta = compute_clip_target_shift(clean_x, protected_x, TARGET_CONCEPT)

protected_loader = DataLoader(TensorDataset(protected_x, clean_y), batch_size=BATCH_SIZE, shuffle=True)
print("Training victim on concept-poisoned Caltech-101 train set...")
victim, victim_history = train_classifier_with_history(
    lambda: get_victim_resnet18(num_classes=num_classes, device=device),
    protected_loader,
    VICTIM_EPOCHS,
    device,
)
clean_acc, asr = compute_attack_success_rate(victim, test_loader, device)

torch.save(
    {
        "model_state_dict": baseline.state_dict(),
        "model": "ResNet-18",
        "dataset": "Caltech-101",
        "train_data": "clean",
        "epochs": BASELINE_EPOCHS,
        "accuracy": baseline_acc,
        "run_name": RUN_NAME,
    },
    MODEL_DIR / "baseline_clean_model.pt",
)
torch.save(
    {
        "model_state_dict": victim.state_dict(),
        "model": "ResNet-18",
        "dataset": "Caltech-101",
        "train_data": "protected_concept_poisoning",
        "epochs": VICTIM_EPOCHS,
        "accuracy": clean_acc,
        "run_name": RUN_NAME,
    },
    MODEL_DIR / "victim_protected_model.pt",
)

metrics = {
    "technique": "concept_poisoning",
    "dataset": "Caltech-101",
    "victim_model": "ResNet-18",
    "num_classes": num_classes,
    "subset_size": SUBSET_SIZE,
    "epsilon": EPSILON,
    "target_concept": TARGET_CONCEPT,
    "baseline_epochs": BASELINE_EPOCHS,
    "victim_epochs": VICTIM_EPOCHS,
    "pgd_steps": PGD_STEPS,
    "baseline_clean_test_accuracy": baseline_acc,
    "protected_clean_test_accuracy": clean_acc,
    "accuracy_drop": round(baseline_acc - clean_acc, 4),
    "asr_proxy": asr,
    "clip_target_similarity_before": clip_before,
    "clip_target_similarity_after": clip_after,
    "clip_target_similarity_delta": clip_delta,
    "psnr": compute_psnr(clean_x, protected_x),
    "ssim": compute_ssim(clean_x, protected_x),
    "linf": compute_linf(clean_x, protected_x),
    "run_name": RUN_NAME,
    "run_dir": str(RUN_DIR),
}
print(metrics)

pd.DataFrame([metrics]).to_csv(TABLE_DIR / "concept_poisoning_caltech101_experiment.csv", index=False)
pd.DataFrame(baseline_history).to_csv(TABLE_DIR / "baseline_training_history.csv", index=False)
pd.DataFrame(victim_history).to_csv(TABLE_DIR / "protected_training_history.csv", index=False)
# Compatibility copy for quick lookup across runs.
os.makedirs("results/tables", exist_ok=True)
pd.DataFrame([metrics]).to_csv("results/tables/concept_poisoning_caltech101_experiment.csv", index=False)

sample_idx = 0
fig = plot_before_after(clean_x[sample_idx], protected_x[sample_idx], "concept_poisoning_caltech101", save=False)
fig.savefig(FIGURE_DIR / "before_after_concept_poisoning_caltech101.png", dpi=150)
to_pil = torchvision.transforms.ToPILImage()
num_samples_to_save = min(20, clean_x.size(0))
for idx in range(num_samples_to_save):
    to_pil(clean_x[idx]).save(SAMPLE_DIR / f"original_{idx:03d}.png")
    to_pil(protected_x[idx]).save(SAMPLE_DIR / f"protected_{idx:03d}.png")
# Compatibility copies for existing report paths.
os.makedirs("results/protected_samples", exist_ok=True)
to_pil(clean_x[sample_idx]).save("results/protected_samples/concept_poisoning_caltech101_original.png")
to_pil(protected_x[sample_idx]).save("results/protected_samples/concept_poisoning_caltech101_protected.png")

print("Saved run to:", RUN_DIR)
print("Key files:")
print("-", TABLE_DIR / "concept_poisoning_caltech101_experiment.csv")
print("-", TENSOR_DIR / "protected_dataset.pt")
print("-", MODEL_DIR / "baseline_clean_model.pt")
print("-", MODEL_DIR / "victim_protected_model.pt")


## 4. Optional copy results to Drive

In [ ]:
# Copy local results to Drive results folder if needed.
# Most experiment code writes to ./results first; this keeps a persistent copy in Drive.
if USE_GOOGLE_DRIVE:
    import shutil
    target = Path(RESULTS_ROOT)
    target.mkdir(parents=True, exist_ok=True)
    shutil.copytree("results", target, dirs_exist_ok=True)
    print("Copied local ./results to", target)
